# Pre-processing & Feature Engineering

Goal of this notebook: Conduct data preprocessing & feature engineering for classification/segmentation tasks. Choices of features are backed up with findings from EDA. 

Major sections:
1. Drop unnecessary columns based on EDA findings.
2. Feature engineering.
3. Train-test split for modeling.
4. Pre-processing pipeline including log-transform, encoding and scaling

Refer to Reports.pdf and decisions.md for detailed findings and decision making process.

In [17]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

DATA_DIR = Path("../data")
SEED = 42

df = pd.read_parquet(DATA_DIR / "df_loaded.parquet")
print(df.shape)
df.head()

(199523, 43)


,age,class of worker,detailed industry recode,detailed occupation recode,education,wage per hour,enroll in edu inst last wk,marital stat,major industry code,major occupation code,race,hispanic origin,sex,member of a labor union,reason for unemployment,full or part time employment stat,capital gains,capital losses,dividends from stocks,tax filer stat,region of previous residence,state of previous residence,detailed household and family stat,detailed household summary in household,weight,migration code-change in msa,migration code-change in reg,migration code-move within reg,live in this house 1 year ago,migration prev res in sunbelt,num persons worked for employer,family members under 18,country of birth father,country of birth mother,country of birth self,citizenship,own business or self employed,fill inc questionnaire for veteran's admin,veterans benefits,weeks worked in year,year,label,y
0,73,Not in universe,0,0,High school graduate,0,Not in universe,Widowed,Not in universe or children,Not in universe,White,All other,Female,Not in universe,Not in universe,Not in labor force,0,0,0,Nonfiler,Not in universe,Not in universe,Other Rel 18+ ever marr not in subfamily,Other relative of householder,1700.09,?,?,?,Not in universe under 1 year old,?,0,Not in universe,United-States,United-States,United-States,Native- Born in the United States,0,Not in universe,2,0,95,- 50000.,0
1,58,Self-employed-not incorporated,4,34,Some college but no degree,0,Not in universe,Divorced,Construction,Precision production craft & repair,White,All other,Male,Not in universe,Not in universe,Children or Armed Forces,0,0,0,Head of household,South,Arkansas,Householder,Householder,1053.55,MSA to MSA,Same county,Same county,No,Yes,1,Not in universe,United-States,United-States,United-States,Native- Born in the United States,0,Not in universe,2,52,94,- 50000.,0
2,18,Not in universe,0,0,10th grade,0,High school,Never married,Not in universe or children,Not in universe,Asian or Pacific Islander,All other,Female,Not in universe,Not in universe,Not in labor force,0,0,0,Nonfiler,Not in universe,Not in universe,Child 18+ never marr Not in a subfamily,Child 18 or older,991.95,?,?,?,Not in universe under 1 year old,?,0,Not in universe,Vietnam,Vietnam,Vietnam,Foreign born- Not a citizen of U S,0,Not in universe,2,0,95,- 50000.,0
3,9,Not in universe,0,0,Children,0,Not in universe,Never married,Not in universe or children,Not in universe,White,All other,Female,Not in universe,Not in universe,Children or Armed Forces,0,0,0,Nonfiler,Not in universe,Not in universe,Child <18 never marr not in subfamily,Child under 18 never married,1758.14,Nonmover,Nonmover,Nonmover,Yes,Not in universe,0,Both parents present,United-States,United-States,United-States,Native- Born in the United States,0,Not in universe,0,0,94,- 50000.,0
4,10,Not in universe,0,0,Children,0,Not in universe,Never married,Not in universe or children,Not in universe,White,All other,Female,Not in universe,Not in universe,Children or Armed Forces,0,0,0,Nonfiler,Not in universe,Not in universe,Child <18 never marr not in subfamily,Child under 18 never married,1069.16,Nonmover,Nonmover,Nonmover,Yes,Not in universe,0,Both parents present,United-States,United-States,United-States,Native- Born in the United States,0,Not in universe,0,0,94,- 50000.,0


### Drop unncessary columns

In [18]:
# To-be-dropped columns due to low MI
low_mi_drops = [
    "migration code-change in msa",
    "region of previous residence",
    "live in this house 1 year ago",
    "year",
    "migration prev res in sunbelt",
    "migration code-move within reg",
    "migration code-change in reg"
]

# To-be-dropped columns due to information redundancy
redundant_drops = [
    "major industry code",     
    "major occupation code",   
]

drop_cols = list(set(low_mi_drops + redundant_drops + ["label"]))
df_pp = df.drop(columns=drop_cols)
print(f"Dropped {len(drop_cols)} columns; {df_pp.shape[1]} remaining")

Dropped 10 columns; 33 remaining


### Feature Engineering

In [ ]:
# From EDA: capital gains/losses/dividends are heavily zero-inflated,
# and the nonzero subgroup has much-higher-than-baseline positive rates.
# Add a binary "has nonzero" flag for our baseline linear model later
zero_inflated = ["capital gains", "capital losses", "dividends from stocks"]
for c in zero_inflated:
    df_pp[f"{c}_nonzero"] = (df_pp[c] > 0).astype(int)

In [20]:
df_pp['education'].unique()

array(['High school graduate', 'Some college but no degree', '10th grade',
       'Children', 'Bachelors degree(BA AB BS)',
       'Masters degree(MA MS MEng MEd MSW MBA)', 'Less than 1st grade',
       'Associates degree-academic program', '7th and 8th grade',
       '12th grade no diploma', 'Associates degree-occup /vocational',
       'Prof school degree (MD DDS DVM LLB JD)', '5th or 6th grade',
       '11th grade', 'Doctorate degree(PhD EdD)', '9th grade',
       '1st 2nd 3rd or 4th grade'], dtype=object)

In [21]:
# Education ordinal
education_order = {
    "Children": 0,
    "Less than 1st grade": 1,
    "1st 2nd 3rd or 4th grade": 2,
    "5th or 6th grade": 3,
    "7th and 8th grade": 4,
    "9th grade": 5,
    "10th grade": 6,
    "11th grade": 7,
    "12th grade no diploma": 8,
    "High school graduate": 9,
    "Some college but no degree": 10,
    "Associates degree-occup /vocational": 11,
    "Associates degree-academic program": 11,
    "Bachelors degree(BA AB BS)": 12,
    "Masters degree(MA MS MEng MEd MSW MBA)": 13,
    "Prof school degree (MD DDS DVM LLB JD)": 14,
    "Doctorate degree(PhD EdD)": 15,
}

unmapped = set(df_pp["education"].dropna().unique()) - set(education_order.keys())
assert not unmapped, f"Unmapped education values, add to dict: {unmapped}"

df_pp["education_ordinal"] = df_pp["education"].map(education_order).astype("int8")
df_pp = df_pp.drop(columns=['education'])

In [22]:
# Employment indicator
df_pp["is_employed"] = (df_pp["weeks worked in year"] > 0).astype("int8")

# positive rate among employed vs not
emp_rate  = df_pp.loc[df_pp["is_employed"] == 1, "y"].mean()
unemp_rate = df_pp.loc[df_pp["is_employed"] == 0, "y"].mean()
print(f"positive rate | employed:     {emp_rate:.4f}")
print(f"positive rate | not employed: {unemp_rate:.4f}")

positive rate | employed:     0.1138
positive rate | not employed: 0.0062


In [23]:
# Number of income sources
df_pp["n_income_sources"] = (
    df_pp["is_employed"]
    + df_pp["capital gains_nonzero"].astype("int8")
    + df_pp["dividends from stocks_nonzero"].astype("int8")
)
print(df_pp["n_income_sources"].value_counts().sort_index())

n_income_sources
0    89427
1    89790
2    18648
3     1658
Name: count, dtype: int64


In [24]:
# Self-employment flag
df_pp["is_self_employed"] = df_pp["class of worker"].str.contains(
    "Self-employed", case=False, na=False
).astype("int8")
print(f"self-employed share: {df_pp['is_self_employed'].mean():.4f}")

self-employed share: 0.0587


In [25]:
# Married joint filer
df_pp["married_joint_filer"] = df_pp["tax filer stat"].str.contains(
    "Joint", case=False, na=False
).astype("int8")

print(f"joint filer share: {df_pp['married_joint_filer'].mean():.4f}")
print(f"positive rate | joint filer:     {df_pp.loc[df_pp['married_joint_filer']==1, 'y'].mean():.4f}")
print(f"positive rate | not joint filer: {df_pp.loc[df_pp['married_joint_filer']==0, 'y'].mean():.4f}")

joint filer share: 0.3989
positive rate | joint filer:     0.1207
positive rate | not joint filer: 0.0232


In [26]:
df_pp = df_pp.replace("?", np.nan)
print("rows with any NaN:", df_pp.isna().any(axis=1).sum())

rows with any NaN: 9794


### Train-test Split

In [ ]:
# 80:20 Split, with target stratification in consideration of class imbalance
TARGET = "y"
WEIGHT = "weight"

X = df_pp.drop(columns=[TARGET, WEIGHT])
y = df_pp[TARGET]
w = df_pp[WEIGHT]

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, w, test_size=0.2, stratify=y, random_state=SEED
)
print(f"train: {X_train.shape}  test: {X_test.shape}")
print(f"train positive rate: {y_train.mean():.4f}   test positive rate: {y_test.mean():.4f}")

train: (159618, 38)  test: (39905, 38)
train prevalence: 0.0621   test prevalence: 0.0620


In [ ]:
# Glimpse of all features to use
numeric_cols = X_train.select_dtypes(include="number").columns.tolist()
categorical_cols = X_train.select_dtypes(include="object").columns.tolist()

HIGH_CARD_THRESHOLD = 15
low_card_cat  = [c for c in categorical_cols if X_train[c].nunique() <= HIGH_CARD_THRESHOLD]
high_card_cat = [c for c in categorical_cols if X_train[c].nunique() >  HIGH_CARD_THRESHOLD]

print(f"numeric ({len(numeric_cols)}): {numeric_cols}")
print(f"low-card categorical ({len(low_card_cat)}): {low_card_cat}")
print(f"high-card categorical ({len(high_card_cat)}): {high_card_cat}")

numeric (15): ['age', 'wage per hour', 'capital gains', 'capital losses', 'dividends from stocks', 'num persons worked for employer', 'weeks worked in year', 'capital gains_nonzero', 'capital losses_nonzero', 'dividends from stocks_nonzero', 'education_ordinal', 'is_employed', 'n_income_sources', 'is_self_employed', 'married_joint_filer']
low-card cat (16): ['class of worker', 'enroll in edu inst last wk', 'marital stat', 'race', 'hispanic origin', 'sex', 'member of a labor union', 'reason for unemployment', 'full or part time employment stat', 'tax filer stat', 'detailed household summary in household', 'family members under 18', 'citizenship', 'own business or self employed', "fill inc questionnaire for veteran's admin", 'veterans benefits']
high-card cat (7): ['detailed industry recode', 'detailed occupation recode', 'state of previous residence', 'detailed household and family stat', 'country of birth father', 'country of birth mother', 'country of birth self']


In [ ]:
# Pre-processing steps including log transform, scaling and encoding

zero_inflated_cols = ["capital gains", "capital losses", "dividends from stocks"]
other_numeric = [c for c in numeric_cols if c not in zero_inflated_cols]

preprocessor = ColumnTransformer(
    transformers=[
        # Zero-inflated numerics: log1p first, then scale
        ("num_zi", Pipeline([
            ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
            ("scale", StandardScaler()),
        ]), zero_inflated_cols),

        # Other numerics: just scale
        ("num_other", StandardScaler(), other_numeric),

        ("low_card", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), low_card_cat),

        ("high_card", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", TargetEncoder(random_state=SEED, smooth="auto")),
        ]), high_card_cat),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)
preprocessor.set_output(transform="pandas")

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_zi', ...), ('num_other', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_

In [30]:
X_train_pp = preprocessor.fit_transform(X_train, y_train)
X_test_pp  = preprocessor.transform(X_test)

print(f"X_train_pp: {X_train_pp.shape}")
print(f"X_test_pp:  {X_test_pp.shape}")
X_train_pp.head()

X_train_pp: (159618, 108)
X_test_pp:  (39905, 108)


,capital gains,capital losses,dividends from stocks,age,wage per hour,num persons worked for employer,weeks worked in year,capital gains_nonzero,capital losses_nonzero,dividends from stocks_nonzero,education_ordinal,is_employed,n_income_sources,is_self_employed,married_joint_filer,class of worker_Federal government,class of worker_Local government,class of worker_Never worked,class of worker_Not in universe,class of worker_Private,class of worker_Self-employed-incorporated,class of worker_Self-employed-not incorporated,class of worker_State government,class of worker_Without pay,enroll in edu inst last wk_College or university,enroll in edu inst last wk_High school,enroll in edu inst last wk_Not in universe,marital stat_Divorced,marital stat_Married-A F spouse present,marital stat_Married-civilian spouse present,marital stat_Married-spouse absent,marital stat_Never married,marital stat_Separated,marital stat_Widowed,race_Amer Indian Aleut or Eskimo,race_Asian or Pacific Islander,race_Black,race_Other,race_White,hispanic origin_All other,hispanic origin_Central or South American,hispanic origin_Chicano,hispanic origin_Cuban,hispanic origin_Do not know,hispanic origin_Mexican (Mexicano),hispanic origin_Mexican-American,hispanic origin_Other Spanish,hispanic origin_Puerto Rican,hispanic origin_None,sex_Female,...,reason for unemployment_Other job loser,reason for unemployment_Re-entrant,full or part time employment stat_Children or Armed Forces,full or part time employment stat_Full-time schedules,full or part time employment stat_Not in labor force,full or part time employment stat_PT for econ reasons usually FT,full or part time employment stat_PT for econ reasons usually PT,full or part time employment stat_PT for non-econ reasons usually FT,full or part time employment stat_Unemployed full-time,full or part time employment stat_Unemployed part- time,tax filer stat_Head of household,tax filer stat_Joint both 65+,tax filer stat_Joint both under 65,tax filer stat_Joint one under 65 & one 65+,tax filer stat_Nonfiler,tax filer stat_Single,detailed household summary in household_Child 18 or older,detailed household summary in household_Child under 18 ever married,detailed household summary in household_Child under 18 never married,detailed household summary in household_Group Quarters- Secondary individual,detailed household summary in household_Householder,detailed household summary in household_Nonrelative of householder,detailed household summary in household_Other relative of householder,detailed household summary in household_Spouse of householder,family members under 18_Both parents present,family members under 18_Father only present,family members under 18_Mother only present,family members under 18_Neither parent present,family members under 18_Not in universe,citizenship_Foreign born- Not a citizen of U S,citizenship_Foreign born- U S citizen by naturalization,citizenship_Native- Born abroad of American Parent(s),citizenship_Native- Born in Puerto Rico or U S Outlying,citizenship_Native- Born in the United States,own business or self employed_0,own business or self employed_1,own business or self employed_2,fill inc questionnaire for veteran's admin_No,fill inc questionnaire for veteran's admin_Not in universe,fill inc questionnaire for veteran's admin_Yes,veterans benefits_0,veterans benefits_1,veterans benefits_2,detailed industry recode,detailed occupation recode,state of previous residence,detailed household and family stat,country of birth father,country of birth mother,country of birth self
19304,-0.194375,-0.141615,-0.321501,-1.365988,-0.200987,-0.827535,-0.949845,-0.195861,-0.141726,-0.344821,-1.528228,-1.038936,-0.975846,-0.249727,-0.814971,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.

In [ ]:
# Save the data for modeling
X_train_pp.to_parquet(DATA_DIR / "X_train.parquet", index=False)
X_test_pp.to_parquet(DATA_DIR / "X_test.parquet", index=False)
y_train.to_frame("y").to_parquet(DATA_DIR / "y_train.parquet", index=False)
y_test.to_frame("y").to_parquet(DATA_DIR / "y_test.parquet", index=False)
w_train.to_frame("weight").to_parquet(DATA_DIR / "w_train.parquet", index=False)
w_test.to_frame("weight").to_parquet(DATA_DIR / "w_test.parquet", index=False)

joblib.dump(preprocessor, DATA_DIR / "preprocessor.joblib")
print("Saved preprocessed splits and fitted preprocessor.")

Saved preprocessed splits and fitted preprocessor.
